In [28]:
import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd

from scipy.spatial.distance import cosine
from heapq import nsmallest

from sklearn.decomposition import PCA
from scipy.stats import pearsonr

from skdim.id import FisherS, TLE, MLE, CorrInt, ESS, DANCo, KNN, TwoNN, MOM, lPCA

def load_fmri_data(data_dir='./'):
    raw = sio.loadmat(os.path.join(data_dir, "NKI_281_Yeo_114.mat"))['subjects']
    
    # sort_raw_to_orig_indices
    id_raw = np.zeros(281, dtype=int)
    for i in range(281):
        id_raw[i] = int(raw[0, i][0][0].split("A")[1])
    indices = np.argsort(id_raw)
    

    raw_645 = np.zeros((281, 113, 884))
    for i in range(281):
        raw_645[i, :23] = raw[0, i][3][:23, :884]
        raw_645[i, 23:] = raw[0, i][3][24:, :884]

    raw_zscored = (raw_645 - np.mean(raw_645, axis=-1)[..., np.newaxis]) / np.std(raw_645, axis=-1)[..., np.newaxis]
    raw_zscored = raw_zscored[indices]

    NETWORKS = {
        "VIS": [(0, 4), (56, 60)],
        "SMN": [(5, 9), (61, 65)],
        "DAN": [(10, 16), (66, 72)],
        "VAN": [(17, 26), (73, 85)],
        "LIM": [(27, 28), (86, 87)],
        "CON": [(29, 42), (88, 99)],
        "DMN": [(43, 55), (100, 112)]
    }
    areas = np.zeros(113)
    for i, nw in enumerate(NETWORKS):
        for p in NETWORKS[nw]:
            areas[p[0]:p[1]+1] = i
    
    return raw_zscored, list(NETWORKS.keys()), areas


raw_zscored, keys, areas = load_fmri_data(data_dir='example_data/')

print(raw_zscored.shape)

(281, 113, 884)


In [ ]:

normalized_raw_zscored = np.zeros_like(raw_zscored)

for i in range(len(raw_zscored)):
    # Normalization
    norms = np.linalg.norm(raw_zscored[i], axis=0, keepdims=True)
    normalized_raw_zscored[i] = raw_zscored[i] / norms
    

In [ ]:
mat = sio.loadmat('example_data/Subjects_N281_Behav_Dyn.mat')

data_behav = pd.DataFrame(mat['unnamed']).astype('float64') 

#1. ID Personen
#2. Age
#3. Sex (2 = female)
#4. Hand (-1 links, 1 = rechts)
#5. FSIQ
#11. Mean Framewise Displacement

age = data_behav[1]
iq = data_behav[4]

def shuffle_along_axis(array, axis):
    """
    Shuffle a NumPy array along a specified axis.

    Parameters:
    array (numpy.ndarray): The input array to shuffle.
    axis (int): The axis along which to shuffle.

    Returns:
    numpy.ndarray: The shuffled array.
    """
    # Create an array of indices along the specified axis
    indices = np.arange(array.shape[axis])
    # Shuffle the indices
    np.random.shuffle(indices)
    # Generate slices for advanced indexing
    slices = [slice(None)] * array.ndim
    slices[axis] = indices
    # Apply the shuffled indices to the array
    shuffled_array = array[tuple(slices)]
    return shuffled_array


In [ ]:
for i in range(len(raw_zscored)):
    plt.figure(figsize=(12,5))
    plt.plot(raw_zscored[i].T)
    plt.title(i)
    plt.show()
    

In [ ]:

def compute_avg_min_cosine_distances(data, timeframe, k):
    """
    Compute the average minimal cosine distance to the nearest k patterns 
    for each pattern to preceding and following patterns within a timeframe.

    Args:
        data (np.ndarray): Array of shape (n_timepoints, n_pixels).
        timeframe (int): The number of timepoints to consider before and after.
        k (int): Number of nearest patterns to average.

    Returns:
        tuple: Two arrays of shape (n_timepoints,) containing the average minimal cosine 
               distances to preceding and following patterns for each timepoint.
    """
    n_timepoints, _ = data.shape
    
    # Initialize arrays to store average minimal distances
    avg_min_dist_to_preceding = np.full(n_timepoints, np.inf)
    avg_min_dist_to_following = np.full(n_timepoints, np.inf)
    
    for t in range(n_timepoints):
        # Compute distances to preceding patterns within the timeframe
        preceding_distances = []
        for tp in range(max(0, t - timeframe), t):
            distance = cosine(data[t], data[tp])
            preceding_distances.append(distance)
        
        # Take the average of the smallest k distances
        if preceding_distances:
            k_smallest_preceding = nsmallest(k, preceding_distances)
            avg_min_dist_to_preceding[t] = np.mean(k_smallest_preceding)
        
        # Compute distances to following patterns within the timeframe
        following_distances = []
        for tf in range(t + 1, min(n_timepoints, t + timeframe + 1)):
            distance = cosine(data[t], data[tf])
            following_distances.append(distance)
        
        # Take the average of the smallest k distances
        if following_distances:
            k_smallest_following = nsmallest(k, following_distances)
            avg_min_dist_to_following[t] = np.mean(k_smallest_following)
    
    return avg_min_dist_to_preceding, avg_min_dist_to_following


min_pres = []
min_fol = []
min_shuffled = []


for i in range(len(raw_zscored)):
    min_preceding, min_following = compute_avg_min_cosine_distances(raw_zscored[i].T[::10,:], timeframe=20, k=3)
    
    min_pre_shuffled, min_post_shuffled = compute_avg_min_cosine_distances(shuffle_along_axis(raw_zscored[i].T[::10,:], axis=0), timeframe=20, k=5)

    min_pres.append(min_preceding)
    min_fol.append(min_fol)
    min_shuffled.append(min_pre_shuffled)
    
    plt.figure(figsize=(12, 5))
    plt.plot(min_preceding[1:])
    plt.plot(min_following[:-1])
    
    plt.axhline(np.mean(min_pre_shuffled[1:]), color='black', linestyle='-', )
    plt.axhline(np.mean(min_pre_shuffled[1:]) + np.std(min_pre_shuffled[1:]), color='red', linestyle='--', )
    plt.axhline(np.mean(min_pre_shuffled[1:]) - np.std(min_pre_shuffled[1:]), color='red', linestyle='--',)
    
    plt.ylabel('avg cosine dists k=5')
    plt.xlabel('timeframe')
    plt.title(i)
    plt.show()

In [ ]:
#combined_res = (min_pres, min_fol, min_shuffled)

#np.save('result_data/human_t50_k5.npy', combined_res, allow_pickle=True)

In [ ]:
def k_nearest_neighbors(data, k):
    """
    Finds the k-nearest neighbors for every timeframe in the data.

    Parameters:
        data (numpy.ndarray): Input array of shape (neurons, timeframes).
        k (int): Number of nearest neighbors to find.

    Returns:
        numpy.ndarray: Array of shape (timeframes, k) containing the indices of the k-nearest neighbors.
    """
    # Transpose the data to get shape (timeframes, neurons)
    data_t = data.T  # Shape: (timeframes, neurons)

    # Initialize an array to store the indices of k-nearest neighbors
    num_timeframes = data_t.shape[0]
    knn_indices = np.zeros((num_timeframes, k), dtype=int)

    # Iterate over each timeframe
    for i, point in enumerate(data_t):
        # Compute distances from the current point to all other points
        distances = np.linalg.norm(data_t - point, axis=1)

        # Get the indices of the k smallest distances (excluding itself)
        nearest_neighbors = np.argsort(distances)[1:k+1]  # Skip the first one (itself)

        # Store the indices in the result array
        knn_indices[i] = nearest_neighbors

    return knn_indices  

# Example usage
k = 3
k_indices = []



for i in range(len(raw_zscored)):
    
    data_input = normalized_raw_zscored[i][:,10::15]
    k_nearest = k_nearest_neighbors(data_input, k=k) # Shape: (timeframes, k)
    k_indices.append(k_nearest)
    
    plt.figure(figsize=(12, 5))
    
    plt.plot([0, data_input.shape[1]], [0, data_input.shape[1]], color='gray')
    
    
    for  k_i in range(k):
        plt.scatter(range(data_input.shape[1]), k_nearest[:,k_i])
    
    
    
    plt.ylabel('index of nearest time point')
    plt.xlabel('timeframe')
    plt.title(i)
    plt.show()

In [ ]:
raw_zscored.shape

In [ ]:

def effective_dimensionality(eigenvalues):
    """
    Compute the effective dimensionality using the participation ratio.
    Formula: (sum(eigenvalues)^2) / sum(eigenvalues^2)
    """
    return (np.sum(eigenvalues) ** 2) / np.sum(eigenvalues ** 2)

# Example function to compute effective dimensionality for incremental timeframes
def compute_effective_dimensionality(data, step=1):
    """
    Compute the effective dimensionality of neural patterns for incrementally increasing subsets of timeframes.

    Parameters:
        data (ndarray): Array of shape (n_neurons, t_timeframes)
        step (int): Increment step for timeframes

    Returns:
        list: Effective dimensionality for each subset of timeframes
    """
    n_neurons, t_timeframes = data.shape
    results = []

    for t in range(10, t_timeframes + 1, step):
        subset = data[:, :t]  # Take subset of timeframes
        
        # Perform PCA
        pca = PCA()
        pca.fit(subset.T)  # Transpose to have shape (timeframes, n_neurons)

        # Get eigenvalues (explained variance)
        eigenvalues = pca.explained_variance_

        # Compute effective dimensionality
        eff_dim = effective_dimensionality(eigenvalues)
        results.append(eff_dim)

    return results


def compute_effective_dimensionality_window(data, window_size, step=1):
    """
    Compute the effective dimensionality of neural patterns within a moving window along the time axis.

    Parameters:
        data (ndarray): Array of shape (n_neurons, t_timeframes)
        window_size (int): Size of the moving window
        step (int): Step size for moving the window

    Returns:
        list: Effective dimensionality for each window position
    """
    n_neurons, t_timeframes = data.shape
    results = []

    for start in range(0, t_timeframes - window_size + 1, step):
        subset = data[:, start:start + window_size]  # Take subset of timeframes
        
        # Perform PCA
        pca = PCA()
        pca.fit(subset.T)  # Transpose to have shape (timeframes, n_neurons)

        # Get eigenvalues (explained variance)
        eigenvalues = pca.explained_variance_

        # Compute effective dimensionality
        eff_dim = effective_dimensionality(eigenvalues)
        results.append(eff_dim)

    return results



# Compute effective dimensionality
step = 15  # Increment step for timeframes

dimensionalities = []
shuffled_control_dims = []

for i in range(len(raw_zscored)):
    
    
    effective_dims = compute_effective_dimensionality(normalized_raw_zscored[i][:,::step], step=1)
    print(np.sum(np.abs([value for value in np.diff(effective_dims) if value < 0])))
    print('---')
    
    shuffled_dims_per_person = []
    
    for control in range(10):
        shuffled_dims = compute_effective_dimensionality(shuffle_along_axis(normalized_raw_zscored[i][:,::step], axis=1), step=1)
        shuffled_dims_per_person.append(shuffled_dims)
        plt.plot(shuffled_dims, color='gray')
        print(np.sum(np.abs([value for value in np.diff(shuffled_dims) if value < 0])))

    plt.plot(effective_dims, color='red')
    plt.title(f'increment. eff. Dim.;  index {i}')
    plt.xlabel(f'timepoints (stepsize={step})')
    plt.ylabel('dimensionality')
    plt.show()
    
    dimensionalities.append(effective_dims)
    shuffled_control_dims.append(shuffled_dims_per_person)

In [ ]:
np.save(f'result_data/human_normalized_dimensionalities_step{step}.npy', dimensionalities, allow_pickle=True)
np.save(f'result_data/human_normalized_dimensionalities_shuffled_control_step{step}.npy', shuffled_control_dims, allow_pickle=True)

In [ ]:
dimensionalities = np.load('result_data/human_normalized_dimensionalities_step30.npy', allow_pickle=True)
shuffled_control_dims = np.load('result_data/human_normalized_dimensionalities_shuffled_control_step30.npy', allow_pickle=True)

dec_dims = []
dec_dims_control = []

for subject in range(len(dimensionalities)):
    

    effective_dims_diff = np.diff(dimensionalities[subject])
    
    
    #dec_dims.append(np.sum(effective_dims_diff < 0))
    res = np.sum(np.abs([value for value in effective_dims_diff if value < 0]))
    
    #res = np.max(dimensionalities[subject])  +  np.sum(np.abs([value for value in effective_dims_diff if value < 0]))
    
    dec_dims.append(res)
    
    res = []
    
    for i in range(10):
        shuffled_control_dims_diff = np.diff(shuffled_control_dims[subject][i])
        res.append(np.sum(np.abs([value for value in shuffled_control_dims_diff if value < 0])))
        
    dec_dims_control.append(np.mean(res))

In [ ]:
x = age
y = dec_dims
y_control = dec_dims_control


plt.figure(figsize=(8, 6))
#plt.scatter(x, age, color='blue', label='Data points')
plt.title('Scatter Plot with Linear Correlation Coefficient', fontsize=16)
plt.xlabel('Age', fontsize=14)
plt.ylabel('Y', fontsize=14)
plt.grid(alpha=0.3)
plt.legend(fontsize=12)

sns.regplot(x=x, y=y, color='red')
sns.regplot(x=x, y=y_control, color='gray')


correlation, p_value = pearsonr(x, y)

# Annotate the plot with the correlation coefficient
plt.text(0.05, 0.95, f'Correlation Coefficient: {correlation:.2f}', 
         transform=plt.gca().transAxes, fontsize=12, 
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Show the plot
plt.show()

# Print the correlation coefficient
print(f'Linear Correlation Coefficient: {correlation:.2f}')
print(f'P-value: {p_value:.3f}')

In [ ]:
fisher_dims = []

for i in range(len(raw_zscored)):
    
    fishers = FisherS()
    fishers = fishers.fit_pw(raw_zscored[i][:].T)
    pw_ids = fishers.transform_pw(raw_zscored[i][:].T)
    
    plt.figure(figsize=(12, 5))
    plt.plot(pw_ids)
    plt.ylabel('Pointwise FisherS - Dim')
    plt.xlabel('Timeframe')
    plt.show()
    
    fisher_dims.append(pw_ids)
    
    

In [ ]:
#np.save('result_data/human_corrected_fishers_dims_step1.npy', fisher_dims, allow_pickle=True)

In [ ]:
dimensionalities = np.load('result_data/human_corrected_fishers_dims_step1.npy', allow_pickle=True)

dec_dims = []

for subject in range(len(dimensionalities)):
    

    
    res = np.std(dimensionalities[subject])  
    
    dec_dims.append(res)

In [ ]:
x = age
y = dec_dims


plt.figure(figsize=(8, 6))
#plt.scatter(x, age, color='blue', label='Data points')
plt.title('Scatter Plot with Linear Correlation Coefficient', fontsize=16)
plt.xlabel('Age', fontsize=14)
plt.ylabel('Y', fontsize=14)
plt.grid(alpha=0.3)
plt.legend(fontsize=12)

sns.regplot(x=x, y=y)


correlation, p_value = pearsonr(x, y)

# Annotate the plot with the correlation coefficient
plt.text(0.05, 0.95, f'Correlation Coefficient: {correlation:.2f}', 
         transform=plt.gca().transAxes, fontsize=12, 
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Show the plot
plt.show()

# Print the correlation coefficient
print(f'Linear Correlation Coefficient: {correlation:.2f}')
print(f'P-value: {p_value:.3f}')

In [ ]:
dimensionalities_twonns = np.load('result_data/human_twonn_dims_step3.npy', allow_pickle=True)
dimensionalities_fishers = np.load('result_data/human_fishers_dims_step3.npy', allow_pickle=True)

corrs = []

for subject in range(len(dimensionalities_twonns)):
    
    plt.figure(figsize=(12, 5))
    sns.regplot(x=dimensionalities_twonns[subject], y=dimensionalities_fishers[subject])
    plt.ylabel('Pointwise FisherS - Dim')
    plt.xlabel('Timeframe')
    plt.show()
    
    correlation, p_value = pearsonr(dimensionalities_twonns[subject], dimensionalities_fishers[subject])
    
    corrs.append(correlation)

In [ ]:
dims = []

for i in range(len(raw_zscored)):
    
    dim_per_subject =  {}
    
    for dim_est in [FisherS, lPCA, TLE, MLE, CorrInt, TwoNN, MOM]:
        print(dim_est, i)
        if dim_est == lPCA:
            dim_estimator = dim_est(ver='participation_ratio')
        else:
            dim_estimator = dim_est()
        dim = dim_estimator.fit_transform(normalized_raw_zscored[i][:,::3].T)

        dim_per_subject[str(dim_est).replace("'",'').replace('>', '').split('.')[3]] = dim
        
    dims.append(dim_per_subject)
    
    
np.save('result_data/human_normalized_dims_estimates_step3.npy', dims, allow_pickle=True)

In [ ]:
dim_ests = np.load('result_data/human_normalized_dims_estimates_step3.npy', allow_pickle=True)

dec_dims = []

for subject in range(len(dim_ests)):
    
    res = dim_ests[subject]['CorrInt']
    
    dec_dims.append(res)
    
x = age
y = dec_dims


plt.figure(figsize=(8, 6))
#plt.scatter(x, age, color='blue', label='Data points')
plt.title('Scatter Plot with Linear Correlation Coefficient', fontsize=16)
plt.xlabel('Age', fontsize=14)
plt.ylabel('Y', fontsize=14)
plt.grid(alpha=0.3)
plt.legend(fontsize=12)

sns.regplot(x=x, y=y)


correlation, p_value = pearsonr(x, y)

# Annotate the plot with the correlation coefficient
plt.text(0.05, 0.95, f'Correlation Coefficient: {correlation:.2f}', 
         transform=plt.gca().transAxes, fontsize=12, 
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Show the plot
plt.show()

# Print the correlation coefficient
print(f'Linear Correlation Coefficient: {correlation:.2f}')
print(f'P-value: {p_value:.3f}')

In [ ]:
dim_ests = np.load('result_data/human_corrected_dims_estimates_step3.npy', allow_pickle=True)
dimensionalities = np.load('result_data/human_dimensionalities_step15.npy', allow_pickle=True)

dim_e = []
dim_d = []

for subject in range(len(dim_ests)):
    
    res = dim_ests[subject]['lPCA']
    print(res)
    print(dimensionalities[subject][-1])
    print(' --  ')
    dim_e.append(res)
    dim_d.append(dimensionalities[subject][-1])
    
sns.regplot(x=dim_e, y=dim_d)